In [1]:
# Get the Dataset object for a pipeline run that failed
from cirro import DataPortal
portal = DataPortal()
dataset = portal.get_dataset(project="<>", dataset="<>")

In [2]:
# The complete set of logs from the workflow execution
print(dataset.logs)

PW_AWS_REGION=us-west-2
PW_BATCH_JOB_ROLE=arn:aws:iam::<>:role/Cirro-BatchJobRole-9a31492a
PW_COMPUTE_ENV=AWS_BATCH
PW_DATASET=<>
PW_EXECUTOR=NEXTFLOW
PW_ONDEMAND_JOB_QUEUE=arn:aws:batch:us-west-2:<>:job-queue/Cirro-OnDemand-9a31492a
PW_PROJECT=<>
PW_S3_DATASET=s3://<>/datasets/<>
PW_S3_PREPROCESS_SCRIPT=s3://cirro-resources-<>/process/public/nf-core/rnaseq/3/preprocess.py
PW_S3_SESSION_DIR=s3://<>-scratch/workdir/session/<>
PW_S3_TRANSFORM_WORKFLOW=s3://cirro-resources-<>/process/hutch/data-transforms/workflow/
PW_S3_WORK=s3://<>-scratch/workdir
PW_SPOT_JOB_QUEUE=arn:aws:batch:us-west-2:<>:job-queue/Cirro-Spot-9a31492a
PW_WORKFLOW=nf-core/rnaseq
PW_WORKFLOW_SCRIPT=main.nf
PW_WORKFLOW_VERSION=3.23.0
Mon Jun  1 22:36:16 UTC 2026 SETTING UP SHARED FUNCTIONS
Mon Jun  1 22:36:16 UTC 2026 RUNNING WORKFLOW
Mon Jun  1 22:36:16 UTC 2026 Workflow found, running main workflow
Mon Jun  1 22:36:16 UTC 2026 Running Nextflow executor
Mon Jun  1 22:36:16 UTC 2026 Skipping local agent status update
Mo

In [3]:
# Instead of having to look through the logs, we can quickly jump to the primary failed task
failed_task = dataset.primary_failed_task
print(failed_task.summary)

Name:      NFCORE_RNASEQ:RNASEQ:FASTQ_QC_TRIM_FILTER_SETSTRANDEDNESS:FQ_LINT (SRR25183878)
Status:    FAILED
Exit Code: 1
Work Dir:  s3://<>-scratch/workdir/1d/aa46709bec6eca03d3bab52fc34b3e

--- Script ---
#!/usr/bin/env bash -e -u -o pipefail
fq lint \
    --disable-validator P001 \
    SRR25183878_1.fastq.gz SRR25183878_2.fastq.gz > SRR25183878.fq_lint.txt

# capture process environment
set +u
set +e
cd "$NXF_TASK_WORKDIR"

nxf_eval_cmd() {
    {
        IFS=$'\n' read -r -d '' "${1}";
        IFS=$'\n' read -r -d '' "${2}";
        (IFS=$'\n' read -r -d '' _ERRNO_; return ${_ERRNO_});
    } < <((printf '\0%s\0%d\0' "$(((({ shift 2; "${@}"; echo "${?}" 1>&3-; } | tr -d '\0' 1>&4-) 4>&2- 2>&1- | tr -d '\0' 1>&4-) 3>&1- | exit "$(cat)") 4>&1-)" "${?}" 1>&2) 2>&1)
}

echo '' > .command.env
#
nxf_eval_cmd STDOUT STDERR bash -c "fq lint --version | sed 's/fq-lint //; s/ .*//'"
status=$?
if [ $status -eq 0 ]; then
  echo nxf_out_eval_7="$STDOUT" >> .command.env
  echo /nxf_out_eval_7/=exi

In [4]:
# The DataPortalTask object contains:
# - name
# - exit_code
# - logs
# - script
# - inputs
# - output

# It can be useful to inspect the list of files that were provided as inputs to the task
failed_task.inputs

[WorkDirFile(name='SRR25183878_1.fastq.gz'),
 WorkDirFile(name='SRR25183878_2.fastq.gz')]

In [5]:
# Since the error logs pointed to a particular file, we can use the corresponding
# WorkDirFile object to read it directly.
# Supported functions: .read(), .read_csv(), .readlines(), .read_json()
input_fasta = failed_task.inputs[1]
print("\n".join(input_fasta.readlines(compression="gzip")[:10]))

@SRR25183878.1 NB501108:242:HCTFHAFX3:1:11101:8100:1043 length=76
NNGACTGACCGATAGTGAACCAGTACCGTGAGGGAAAGGCGAAAAGAACCCCGGCGAGGGGAGTGAAAAAGAACCT
+SRR25183878.1 NB501108:242:HCTFHAFX3:1:11101:8100:1043 length=76
##AAAEEAEAEEEEEEEE//EEEEEEEEEEEEEEEEEE6EEEEAEEEEEE6EEA<EEEA/AAE6EEEEEEAEEEEE
-----
--MALFORMED INPUT FILE--
------
@SRR25183878.2 NB501108:242:HCTFHAFX3:1:11101:9709:1043 length=76
NNTCGCCATTCAGGCAGCGATTGGTACGCACATTATTGCGCGATCCACCGTGAAACAGCTGCGTAAAAACGTACTG
+SRR25183878.2 NB501108:242:HCTFHAFX3:1:11101:9709:1043 length=76


In [6]:
# Since we can see that the input file was corrupted, we have since repeated the run
# with that file omitted.
repeat_run = portal.get_dataset(project="<>", dataset="<>")

# This gives us an opportunity to see how it can be possible to walk through the
# chain of processes that are upstream of any step in the workflow, which can often be
# helpful in diagnosing the root cause of an error.

# e.g. Get one of the final tasks in this successful run
repeat_tasks = repeat_run.tasks
final_task = repeat_tasks[-25]
print(final_task.summary)

Name:      NFCORE_RNASEQ:RNASEQ:BEDGRAPH_BEDCLIP_BEDGRAPHTOBIGWIG_REVERSE:UCSC_BEDGRAPHTOBIGWIG (SRR25183879)
Status:    COMPLETED
Exit Code: 0
Work Dir:  s3://<>-scratch/workdir/07/dc6471f743183d035a3328cda80406

--- Script ---
#!/usr/bin/env bash -e -u -o pipefail
bedGraphToBigWig \
     \
    SRR25183879.clip.reverse.bedGraph \
    genome.fa.sizes \
    SRR25183879.reverse.bigWig

# capture process environment
set +u
set +e
cd "$NXF_TASK_WORKDIR"

nxf_eval_cmd() {
    {
        IFS=$'\n' read -r -d '' "${1}";
        IFS=$'\n' read -r -d '' "${2}";
        (IFS=$'\n' read -r -d '' _ERRNO_; return ${_ERRNO_});
    } < <((printf '\0%s\0%d\0' "$(((({ shift 2; "${@}"; echo "${?}" 1>&3-; } | tr -d '\0' 1>&4-) 4>&2- 2>&1- | tr -d '\0' 1>&4-) 3>&1- | exit "$(cat)") 4>&1-)" "${?}" 1>&2) 2>&1)
}

echo '' > .command.env
#
nxf_eval_cmd STDOUT STDERR bash -c "echo 469"
status=$?
if [ $status -eq 0 ]; then
  echo nxf_out_eval_36="$STDOUT" >> .command.env
  echo /nxf_out_eval_36/=exit:0 >> .comma

In [7]:
# Look at the files that went into that task
final_task.inputs

[WorkDirFile(name='genome.fa.sizes'),
 WorkDirFile(name='SRR25183879.clip.reverse.bedGraph')]

In [8]:
# Get the task that produced the bedGraph temporary file
prev_task = final_task.inputs[1].source_task
print(prev_task.summary)

Name:      NFCORE_RNASEQ:RNASEQ:BEDGRAPH_BEDCLIP_BEDGRAPHTOBIGWIG_REVERSE:UCSC_BEDCLIP (SRR25183879)
Status:    COMPLETED
Exit Code: 0
Work Dir:  s3://<>-scratch/workdir/b4/41832387b29b6711bd3ff82c981a19

--- Script ---
#!/usr/bin/env bash -e -u -o pipefail
bedClip \
     \
    SRR25183879.reverse.bedGraph \
    genome.fa.sizes \
    SRR25183879.clip.reverse.bedGraph

# capture process environment
set +u
set +e
cd "$NXF_TASK_WORKDIR"

nxf_eval_cmd() {
    {
        IFS=$'\n' read -r -d '' "${1}";
        IFS=$'\n' read -r -d '' "${2}";
        (IFS=$'\n' read -r -d '' _ERRNO_; return ${_ERRNO_});
    } < <((printf '\0%s\0%d\0' "$(((({ shift 2; "${@}"; echo "${?}" 1>&3-; } | tr -d '\0' 1>&4-) 4>&2- 2>&1- | tr -d '\0' 1>&4-) 3>&1- | exit "$(cat)") 4>&1-)" "${?}" 1>&2) 2>&1)
}

echo '' > .command.env
#
nxf_eval_cmd STDOUT STDERR bash -c "echo 377"
status=$?
if [ $status -eq 0 ]; then
  echo nxf_out_eval_35="$STDOUT" >> .command.env
  echo /nxf_out_eval_35/=exit:0 >> .command.env
else
  ec